In [1]:
# import packages
using Ferrite
using SparseArrays
using WriteVTK

# declare variables
Ca = 59.187
Cd = 9.57
Ea = 21179.6
Ed = 16473
eps = 0.5
rho_emp = 7164
rho_sat = 7259
R = 8.314
Aa = 10.7
Ba = 3704.6
Ad = 10.57
Bd = 3704.6
Ta = 301.15
Td = 305.15
Pas = [10., 15., 20.]
Pds = [1.5, 1.75, 2.]
Pref = 10
Peqa = Pref * exp((Aa - Ba / Ta))
Peqd = Pref * exp((Ad - Bd / Td))
uxs = [0.1, 1., 10.]
D = 1.
L = 0.8
H = 0.04

# initialize grid and trial/test functions
N = 10
grid = generate_grid(Quadrilateral, (N, N), Vec{2}((0.0, 0.0)), Vec{2}((L, H)))
ip = Lagrange{RefQuadrilateral, 1}()
qr = QuadratureRule{RefQuadrilateral}(2)
cellvalues = CellValues(qr, ip)

# handle degrees of freedom
dh = DofHandler(grid)
add!(dh, :u, ip)
close!(dh)

# define assembly functions
function steady_state_assemble_element!(Ke::Matrix, cellvalues::CellValues, v)
    n_basefuncs = getnbasefunctions(cellvalues)
    # Reset to 0
    fill!(Ke, 0)
    # Loop over quadrature points
    for q_point in 1:getnquadpoints(cellvalues)
        # Get the quadrature weight
        dΩ = getdetJdV(cellvalues, q_point)
        # Loop over test shape functions
        for i in 1:n_basefuncs
            δu = shape_value(cellvalues, q_point, i)
            ∇δu = shape_gradient(cellvalues, q_point, i)
            # Loop over trial shape functions
            for j in 1:n_basefuncs
                ∇u = shape_gradient(cellvalues, q_point, j)
                # Add contribution to Ke
                Ke[i, j] += (- D * (∇δu ⋅ ∇u) + (v ⋅ ∇u) * δu) * dΩ
            end
        end
    end
    return Ke
end

function steady_state_assemble_global!(cellvalues::CellValues, K::SparseMatrixCSC, dh::DofHandler, v)
    # Allocate the element stiffness matrix
    n_basefuncs = getnbasefunctions(cellvalues)
    Ke = zeros(n_basefuncs, n_basefuncs)
    # Create an assembler
    assembler = start_assemble(K)
    # Loop over all cells
    for cell in CellIterator(dh)
        # Reinitialize cellvalues for this cell
        reinit!(cellvalues, cell)
        # Compute element contribution
        steady_state_assemble_element!(Ke, cellvalues, v)
        # Assemble Ke into K
        assemble!(assembler, celldofs(cell), Ke)
    end
    return K
end

function assemble_element!(Ke::Matrix, fe::Vector, cellvalues::CellValues, v, rho_s, mdot_const, absorption)
    n_basefuncs = getnbasefunctions(cellvalues)
    # Reset to 0
    fill!(Ke, 0)
    fill!(fe, 0)
    # Loop over quadrature points
    for q_point in 1:getnquadpoints(cellvalues)
        # Get the quadrature weight
        dΩ = getdetJdV(cellvalues, q_point)
        # Loop over test shape functions
        for i in 1:n_basefuncs
            δu = shape_value(cellvalues, q_point, i)
            ∇δu = shape_gradient(cellvalues, q_point, i)
            # Add contribution to fe
            if absorption == true
                mdot = mdot_const * (rho_sat - rho_s)
                fe[i] += mdot * δu * dΩ
            elseif absorption == false
                mdot = mdot_const * (rho_s - rho_emp)
                fe[i] += mdot * δu * dΩ
            end
            # Loop over trial shape functions
            for j in 1:n_basefuncs
                ∇u = shape_gradient(cellvalues, q_point, j)
                # Add contribution to Ke
                Ke[i, j] += (- D * (∇δu ⋅ ∇u) + (v ⋅ ∇u) * δu) * dΩ
            end
        end
    end
    return Ke, fe
end

function assemble_global!(cellvalues::CellValues, K::SparseMatrixCSC, dh::DofHandler, v, rho_s, mdot_const, absorption)
    # Allocate the element stiffness matrix and element force vector
    n_basefuncs = getnbasefunctions(cellvalues)
    Ke = zeros(n_basefuncs, n_basefuncs)
    fe = zeros(n_basefuncs)
    # Allocate global force vector f
    f = zeros(ndofs(dh))
    # Create an assembler
    assembler = start_assemble(K, f)
    # Loop over all cells
    for cell in CellIterator(dh)
        # Reinitialize cellvalues for this cell
        reinit!(cellvalues, cell)
        # Compute element contribution
        assemble_element!(Ke, fe, cellvalues, v, rho_s, mdot_const, absorption)
        # Assemble Ke and fe into K and f
        assemble!(assembler, celldofs(cell), Ke, fe)
    end
    return K, f
end

function assemble_M!(M::SparseMatrixCSC, cellvalues::CellValues, dh::DofHandler)

    n_basefuncs = getnbasefunctions(cellvalues)
    Me = zeros(n_basefuncs, n_basefuncs)

    assembler = start_assemble(M)

    for cell in CellIterator(dh)

        fill!(Me, 0)

        reinit!(cellvalues, cell)

        for q_point in 1:getnquadpoints(cellvalues)
            dΩ = getdetJdV(cellvalues, q_point)

            for i in 1:n_basefuncs
                δu = shape_value(cellvalues, q_point, i)
                for j in 1:n_basefuncs
                    u = shape_value(cellvalues, q_point, j)
                    Me[i, j] += δu * u * dΩ
                end
            end
        end

        assemble!(assembler, celldofs(cell), Me)
    end
    return M
end;

## 2D Reactor Models with A-Priori Given Axial Velocity 

### Steady-State Model: Convection-Diffusion Equation for Gas Density

<b>Absorption, Dirichlet Boundary Conditions<b>

In [2]:
# choose velocity
v = Vec{2}((-uxs[3], 0.))   # uxs = [0.1, 1., 10.]

# account for dirichlet boundary conditions
ch = ConstraintHandler(dh)
dbc = Dirichlet(:u, getfacetset(grid, "left"), (x, t) -> rho_sat - rho_emp)
add!(ch, dbc)
dbc = Dirichlet(:u, getfacetset(grid, "right"), (x, t) -> 0.)
add!(ch, dbc)
close!(ch)

# assemble linear system
K = allocate_matrix(dh)
K = steady_state_assemble_global!(cellvalues, K, dh, v)
f = zeros(ndofs(dh))
apply!(K, f, ch)

# solve
u = K \ f

# export to VTK
if isfile("gas_dirichlet_steady_absorption.vtu")
    rm("gas_dirichlet_steady_absorption.vtu")
end
VTKGridFile("gas_dirichlet_steady_absorption", dh) do vtk
    write_solution(vtk, dh, u)
end

VTKGridFile for the closed file "gas_dirichlet_steady_absorption.vtu".

<b>Absorption, Dirichlet and Neumann Boundary Conditions<b>

In [3]:
# choose velocity
v = Vec{2}((-uxs[3], 0.))   # uxs = [0.1, 1., 10.]

# account for dirichlet boundary condition
ch = ConstraintHandler(dh)
dbc = Dirichlet(:u, getfacetset(grid, "left"), (x, t) -> rho_sat - rho_emp)
add!(ch, dbc)
close!(ch)

# assemble linear system
K = allocate_matrix(dh)
K = steady_state_assemble_global!(cellvalues, K, dh, v)
f = zeros(ndofs(dh))
apply!(K, f, ch)

# solve
u = K \ f

# export to VTK
if isfile("gas_neumann_steady_absorption.vtu")
    rm("gas_neumann_steady_absorption.vtu")
end
VTKGridFile("gas_neumann_steady_absorption", dh) do vtk
    write_solution(vtk, dh, u)
end

VTKGridFile for the closed file "gas_neumann_steady_absorption.vtu".

<b>Desorption, Dirichlet Boundary Conditions<b>

In [4]:
# choose velocity
v = Vec{2}((uxs[3], 0.))   # uxs = [0.1, 1., 10.]

# account for dirichlet boundary conditions
ch = ConstraintHandler(dh)
dbc = Dirichlet(:u, getfacetset(grid, "left"), (x, t) -> rho_sat - rho_emp)
add!(ch, dbc)
dbc = Dirichlet(:u, getfacetset(grid, "right"), (x, t) -> 0.)
add!(ch, dbc)
close!(ch)

# assemble linear system
K = allocate_matrix(dh)
K = steady_state_assemble_global!(cellvalues, K, dh, v)
f = zeros(ndofs(dh))
apply!(K, f, ch)

# solve
u = K \ f

# export to VTK
if isfile("gas_dirichlet_steady_desorption.vtu")
    rm("gas_dirichlet_steady_desorption.vtu")
end
VTKGridFile("gas_dirichlet_steady_desorption", dh) do vtk
    write_solution(vtk, dh, u)
end

VTKGridFile for the closed file "gas_dirichlet_steady_desorption.vtu".

<b>Desorption, Dirichlet and Neumann Boundary Conditions<b>

In [5]:
# choose velocity
v = Vec{2}((uxs[3], 0.))   # uxs = [0.1, 1., 10.]

# account for dirichlet boundary condition
ch = ConstraintHandler(dh)
dbc = Dirichlet(:u, getfacetset(grid, "left"), (x, t) -> 0.)
add!(ch, dbc)
close!(ch)

# assemble linear system
K = allocate_matrix(dh)
K = steady_state_assemble_global!(cellvalues, K, dh, v)
f = zeros(ndofs(dh))
apply!(K, f, ch)

# solve
u = K \ f

# export to VTK
if isfile("gas_neumann_steady_desorption.vtu")
    rm("gas_neumann_steady_desorption.vtu")
end
VTKGridFile("gas_neumann_steady_desorption", dh) do vtk
    write_solution(vtk, dh, u)
end

VTKGridFile for the closed file "gas_neumann_steady_desorption.vtu".

### Transient Model: Coupled Reaction-Convection-Diffusion for Solid Density and Gas Density

<b>Absorption, Dirichlet and Neumann Boundary Conditions<b>

In [6]:
# choose variables
v = Vec{2}((-uxs[3], 0.))   # uxs = [0.1, 1., 10.]
Pa = Pas[1]                 # Pas = [10., 15., 20.]
mdot_const = Ca * exp(-Ea / (R * Ta)) * log(Pa / Peqa)
Δt = 0.5
T = 75

# initialize rho_s
rho_sₙ = zeros(ndofs(dh))
for i in 1:ndofs(dh)
    rho_sₙ[i] = rho_emp
end

# export initial rho_s to VTK
rm.(filter(f -> startswith(f, "solid_transient_absorption"), readdir()))
pvd_s = paraview_collection("solid_transient_absorption")
VTKGridFile("solid_transient_absorption-0", dh) do vtk
    write_solution(vtk, dh, rho_sₙ)
    pvd_s[0.0] = vtk
end

# initialize rho_g
rho_gₙ = zeros(ndofs(dh))

# export initial rho_g to VTK
rm.(filter(f -> startswith(f, "gas_transient_absorption"), readdir()))
pvd_g = paraview_collection("gas_transient_absorption")
VTKGridFile("gas_transient_absorption-0", dh) do vtk
    write_solution(vtk, dh, rho_gₙ)
    pvd_g[0.0] = vtk
end

# initialize f_s
f_s = zeros(ndofs(dh))
for i in 1:ndofs(dh)
    mdot = mdot_const * (rho_sat - rho_emp) 
    f_s[i] = mdot / (1 - eps)
end

# allocate K and assemble M
K = allocate_matrix(dh)
M = allocate_matrix(dh)
M = assemble_M!(M, cellvalues, dh)

# iterate over time
for (step, t) in enumerate(Δt:Δt:T)
    # solve for rho_s
    rho_s = Δt .* f_s .+ rho_sₙ   

    # export to VTK
    VTKGridFile("solid_transient_absorption-$step", dh) do vtk
        write_solution(vtk, dh, rho_s)
        pvd_s[t] = vtk
    end

    # update previous rho_s and f_s
    rho_sₙ .= rho_s
    for i in 1:ndofs(dh)
        mdot = mdot_const * (rho_sat - rho_s[1]) 
        f_s[i] = mdot / (1 - eps)
    end

    # assemble linear system
    K, f_g = assemble_global!(cellvalues, K, dh, v, rho_s[1], mdot_const, true)
    # display(f_g[1])
    A = - Δt * K + eps * M

    # account for dirichlet boundary condition
    ch = ConstraintHandler(dh)
    dbc = Dirichlet(:u, getfacetset(grid, "left"), (x, t) -> rho_s[1] - rho_emp)
    add!(ch, dbc)
    close!(ch)

    # save values relating to boundary conditions
    rhsdata = get_rhs_data(ch, A)
    apply!(A, ch)

    # compute right-hand-side
    b_g = Δt .* f_g .+ eps * M * rho_gₙ
    apply_rhs!(rhsdata, b_g, ch)
    rho_g = A \ b_g

    VTKGridFile("gas_transient_absorption-$step", dh) do vtk
        write_solution(vtk, dh, rho_g)
        pvd_g[t] = vtk
    end

    # update previous rho_g
    rho_gₙ .= rho_g
end

<b>Desorption, Dirichlet and Neumann Boundary Conditions<b>

In [7]:
# choose variables
v = Vec{2}((uxs[3], 0.))    # uxs = [0.1, 1., 10.]
Pd = Pds[1]                 # Pds = [1.5, 1.75, 2.]
mdot_const = Cd * exp(-Ed / (R * Td)) * (Pd - Peqd) / Peqd
Δt = 1
T = 400

# initialize rho_s
rho_sₙ = zeros(ndofs(dh))
for i in 1:ndofs(dh)
    rho_sₙ[i] = rho_sat
end

# export initial rho_s to VTK
rm.(filter(f -> startswith(f, "solid_transient_desorption"), readdir()))
pvd_s = paraview_collection("solid_transient_desorption")
VTKGridFile("solid_transient_desorption-0", dh) do vtk
    write_solution(vtk, dh, rho_sₙ)
    pvd_s[0.0] = vtk
end

# initialize rho_g
rho_gₙ = zeros(ndofs(dh))
for i in 1:ndofs(dh)
    rho_gₙ[i] = rho_sat - rho_emp
end

# export initial rho_g to VTK
rm.(filter(f -> startswith(f, "gas_transient_desorption"), readdir()))
pvd_g = paraview_collection("gas_transient_desorption")
VTKGridFile("gas_transient_desorption-0", dh) do vtk
    write_solution(vtk, dh, rho_gₙ)
    pvd_g[0.0] = vtk
end

# initialize f_s
f_s = zeros(ndofs(dh))
for i in 1:ndofs(dh)
    mdot = mdot_const * (rho_sat - rho_emp) 
    f_s[i] = mdot / (1 - eps)
end

# allocate K and assemble M
K = allocate_matrix(dh)
M = allocate_matrix(dh)
M = assemble_M!(M, cellvalues, dh)

# iterate over time
for (step, t) in enumerate(Δt:Δt:T)
    # solve for rho_s
    rho_s = Δt .* f_s .+ rho_sₙ   

    # export to VTK
    VTKGridFile("solid_transient_desorption-$step", dh) do vtk
        write_solution(vtk, dh, rho_s)
        pvd_s[t] = vtk
    end

    # update previous rho_s and f_s
    rho_sₙ .= rho_s
    for i in 1:ndofs(dh)
        mdot = mdot_const * (rho_s[1] - rho_emp) 
        f_s[i] = mdot / (1 - eps)
    end

    # assemble linear system
    K, f_g = assemble_global!(cellvalues, K, dh, v, rho_s[1], mdot_const, false)
    # display(f_g[1])
    A = - Δt * K + eps * M

    # account for dirichlet boundary condition
    ch = ConstraintHandler(dh)
    dbc = Dirichlet(:u, getfacetset(grid, "left"), (x, t) -> rho_s[1] - rho_emp)
    add!(ch, dbc)
    close!(ch)

    # save values relating to boundary conditions
    rhsdata = get_rhs_data(ch, A)
    apply!(A, ch)

    # compute right-hand-side
    b_g = Δt .* f_g .+ eps * M * rho_gₙ
    apply_rhs!(rhsdata, b_g, ch)
    rho_g = A \ b_g

    VTKGridFile("gas_transient_desorption-$step", dh) do vtk
        write_solution(vtk, dh, rho_g)
        pvd_g[t] = vtk
    end

    # update previous rho_g
    rho_gₙ .= rho_g
end